[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20I%20-%20The%20Port%20%26%20Container%20Terminal%20%28Problems%201-46%29/A.%20Foundations%20of%20Terminal%20Operations%20%28The%20Building%20Blocks%29/06.%20The%20Yard%20Location%20Naming%20Convention%20Problem/P6-Tier-1_executed.ipynb) [![Open in SageMaker](https://img.shields.io/badge/Open%20in-SageMaker-orange?logo=amazon-aws)](https://aws.amazon.com/sagemaker/) [![Open in Azure](https://img.shields.io/badge/Open%20in-Azure%20Notebooks-blue?logo=microsoft-azure)](https://notebooks.azure.com/)

---



# 6. The Yard Location Naming Convention Problem

## Tier 1 — The Pen-and-Paper Method: Mathematical Formulation

### Goal
Construct a systematic naming convention for container-yard locations and assess it using measures of identifier length, pairwise similarity, and access frequency. The example below generates identifiers from a fixed format and evaluates that candidate convention; it does not search over alternative conventions or establish that the candidate is globally optimal.

### Inputs and outputs
- **Inputs:** the yard's blocks, rows, positions, and tiers; the identifier format; access-frequency weights; and the model parameters.
- **Output:** a location-to-identifier mapping, constraint checks, objective-component values, and visual summaries of the generated identifiers.

### Scope and modeling assumptions
- The example yard has 2 blocks, 3 rows per block, 4 positions per row, and 2 tiers per position, for 48 physical locations.
- Every location receives an identifier in the fixed `B-RR-PP-T` format: block, zero-padded row, zero-padded position, and tier.
- Block-level access frequency is used as a proxy for how often a location's identifier is read or used. The model does not represent individual location traffic, travel distance, worker experience, safety, or the operational cost of changing an established naming system.
- The objective is a weighted score for comparing evaluated mappings. Because this notebook constructs one mapping directly from the format, the score is not an optimization proof or a global optimum.

### Approach
1. Define the yard locations and model parameters.
2. Generate identifiers using the stated format.
3. Check uniqueness, maximum length, and minimum pairwise edit distance.
4. Evaluate the objective components, visualize the candidate, and examine how the score depends on the length and similarity weights.

### How to interpret the results
A feasible candidate must give each location a unique identifier, respect the length limit, and meet the required minimum edit distance. A lower objective score indicates a lower value under the chosen mathematical measures and weights; it does not, by itself, establish better real-world usability.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from itertools import combinations

# A consistent plotting style makes the figures easier to compare.
plt.style.use("seaborn-v0_8")

## Mathematical Model Definition

### Sets and parameters
A **location** is represented by a tuple containing its block, row, position, and tier. The location set is generated from the yard dimensions in the next cell. The mapping from a location to an identifier is the candidate naming convention to be assessed.

The model uses three cost components:

1. **Length cost:** longer identifiers receive a squared-length penalty.
2. **Similarity cost:** pairs of identifiers with small Levenshtein distance receive a larger penalty.
3. **Access cost:** longer identifiers are penalized more when their block has higher access frequency.

Let $I$ be the set of locations, let $s_i$ be the identifier assigned to location $i$, and let $\ell_i$ be the number of characters in $s_i$. Let $d_{ij}$ be the Levenshtein distance between identifiers $s_i$ and $s_j$. Let $f_{b(i)}$ denote the access-frequency weight of the block containing location $i$. The score for a candidate mapping is

$$
J = \alpha \sum_{i \in I} \ell_i^2
  + \beta \sum_{\substack{i,j \in I \\ i<j}} \frac{1}{d_{ij}+\varepsilon}
  + \gamma \sum_{i \in I} f_{b(i)}\ell_i.
$$

Here, $\alpha$, $\beta$, and $\gamma$ are nonnegative weights controlling the relative contribution of length, similarity, and access, respectively. The small positive constant $\varepsilon$ prevents division by zero if identical identifiers are compared; uniqueness is also checked separately. The model imposes the constraints $\ell_i \leq L_{\max}$ for every location, distinct identifiers for distinct locations, and $d_{ij} \geq \delta_{\min}$ for every pair of locations.

The function `calculate_objective_value` computes the three unweighted sums and their weighted total. `verify_constraints` checks the feasibility conditions. In this notebook, the mapping is generated from a prescribed format rather than selected by an optimizer.

In [ ]:
# Define the yard structure for the concrete example.
blocks = ["A", "B"]
rows_per_block = {"A": 3, "B": 3}
positions_per_row = 4
tiers_per_position = 2

# Enumerate each physical location once so later checks cover the full yard.
locations = []
for block in blocks:
    for row in range(1, rows_per_block[block] + 1):
        for pos in range(1, positions_per_row + 1):
            for tier in range(1, tiers_per_position + 1):
                locations.append((block, row, pos, tier))

expected_location_count = (
    len(blocks)
    * sum(rows_per_block.values()) / len(blocks)
    * positions_per_row
    * tiers_per_position
)
assert len(locations) == expected_location_count

print(f"Total locations: {len(locations)}")
print(f"Sample locations: {locations[:5]}")

### Decision variables and parameters
The weights express how the three score components are valued relative to one another. The length limit and minimum dissimilarity are feasibility requirements, not objective weights. Access frequency is assigned by block, so the same block-level weight applies to each of its locations.

The candidate mapping is generated in the next section; the parameters below are then passed explicitly to the constraint-checking and objective functions. Passing them as arguments makes clear which values determine each calculation.

In [ ]:
# Objective weights and feasibility thresholds.
alpha = 1.0
beta = 10.0
gamma = 0.1
L_max = 10
delta_min = 1
epsilon = 0.001

# A larger value indicates a block whose locations are accessed more often.
access_frequency = {
    "A": 100,
    "B": 50,
}

print("Model parameters:")
print(f"  Length penalty weight (alpha): {alpha}")
print(f"  Similarity penalty weight (beta): {beta}")
print(f"  Access cost weight (gamma): {gamma}")
print(f"  Maximum identifier length: {L_max}")
print(f"  Minimum edit distance: {delta_min}")
print(f"  Similarity denominator constant (epsilon): {epsilon}")

## Naming Convention Generation

The format `B-RR-PP-T` assigns one character to the block, two zero-padded digits to the row, two zero-padded digits to the position, and one digit to the tier, with hyphens separating the fields. Zero-padding gives row and position numbers a consistent minimum width, which makes the parts of the identifier easier to scan. The field widths must be chosen to accommodate the actual yard dimensions in a larger application.

In [ ]:
def generate_identifier(block, row, pos, tier):
    """Return a location identifier in B-RR-PP-T format.

    B is the block letter; RR and PP are zero-padded row and position
    numbers; T is the tier number.
    """
    return f"{block}-{row:02d}-{pos:02d}-{tier}"


# Applying one explicit format to every location produces the candidate mapping.
location_mapping = {}
for location in locations:
    block, row, pos, tier = location
    location_mapping[location] = generate_identifier(block, row, pos, tier)

print("Sample location identifiers:")
for location, identifier in list(location_mapping.items())[:8]:
    print(f"{location} -> {identifier}")

## Constraint Verification

A naming convention is usable only if it satisfies the model's feasibility requirements. **Uniqueness** prevents two physical locations from sharing an identifier. **Maximum length** limits the number of characters. **Minimum edit distance** prevents identifiers from being too similar under the chosen character-edit measure.

Levenshtein distance is the minimum number of single-character insertions, deletions, and substitutions needed to transform one string into another. Its dynamic-programming calculation builds answers for progressively longer prefixes. If $D(i,j)$ is the distance between the first $i$ characters of one string and the first $j$ characters of the other, then

$$
D(i,0)=i,\qquad D(0,j)=j,
$$

and, for $i,j>0$,

$$
D(i,j)=\min\left\{
D(i-1,j)+1,\;
D(i,j-1)+1,\;
D(i-1,j-1)+\mathbf{1}[x_i\ne y_j]
\right\}.
$$

The three alternatives correspond to a deletion, an insertion, and a match or substitution. The code stores only one row of the dynamic-programming table at a time, so for strings of lengths $m$ and $n$ it uses $O(\min(m,n))$ memory and $O(mn)$ time. The full pairwise check evaluates $\binom{|I|}{2}$ identifier pairs.

In [ ]:
def levenshtein_distance(s1, s2):
    """Return the minimum single-character edit distance between strings."""
    if len(s1) > len(s2):
        s1, s2 = s2, s1

    # Keeping the shorter string in the working row bounds memory use.
    distances = list(range(len(s1) + 1))
    for i2, c2 in enumerate(s2):
        new_distances = [i2 + 1]
        for i1, c1 in enumerate(s1):
            if c1 == c2:
                new_distances.append(distances[i1])
            else:
                new_distances.append(
                    1
                    + min(
                        distances[i1],
                        distances[i1 + 1],
                        new_distances[-1],
                    )
                )
        distances = new_distances
    return distances[-1]


def verify_constraints(location_mapping, L_max, delta_min):
    """Check identifier uniqueness, maximum length, and minimum distance."""
    identifiers = list(location_mapping.values())

    unique_identifiers = set(identifiers)
    uniqueness_satisfied = len(unique_identifiers) == len(identifiers)

    lengths = [len(identifier) for identifier in identifiers]
    max_length = max(lengths, default=0)
    length_satisfied = max_length <= L_max

    min_distance = float("inf")
    for id1, id2 in combinations(identifiers, 2):
        min_distance = min(min_distance, levenshtein_distance(id1, id2))

    similarity_satisfied = min_distance >= delta_min

    return {
        "uniqueness": uniqueness_satisfied,
        "max_length": max_length,
        "length_satisfied": length_satisfied,
        "min_distance": min_distance,
        "similarity_satisfied": similarity_satisfied,
        "total_locations": len(identifiers),
    }


constraint_results = verify_constraints(location_mapping, L_max, delta_min)

print("Constraint verification results:")
print(
    "  Unique identifiers: "
    f"{'Yes' if constraint_results['uniqueness'] else 'No'}"
)
print(
    f"  Maximum identifier length: {constraint_results['max_length']} "
    f"(limit: {L_max}) — "
    f"{'satisfied' if constraint_results['length_satisfied'] else 'not satisfied'}"
)
print(
    f"  Minimum edit distance: {constraint_results['min_distance']} "
    f"(required: at least {delta_min}) — "
    f"{'satisfied' if constraint_results['similarity_satisfied'] else 'not satisfied'}"
)
print(f"  Total locations: {constraint_results['total_locations']}")

## Objective Function Evaluation

The objective score combines the three cost components using the selected weights. The length and access terms are sums over locations; the similarity term is a sum over unordered pairs, so each pair contributes once. The reciprocal-distance term is larger for more similar identifiers, while the access term gives more weight to identifiers in frequently accessed blocks.

The returned component values are unweighted sums. The printed breakdown also shows each weighted contribution, making it possible to verify that the displayed total is the sum of the terms in the mathematical objective.

In [ ]:
def calculate_objective_value(
    location_mapping,
    access_frequency,
    alpha,
    beta,
    gamma,
    epsilon,
):
    """Return unweighted objective components and their weighted total."""
    identifiers = list(location_mapping.values())

    length_cost = sum(len(identifier) ** 2 for identifier in identifiers)

    similarity_cost = 0.0
    for id1, id2 in combinations(identifiers, 2):
        distance = levenshtein_distance(id1, id2)
        similarity_cost += 1 / (distance + epsilon)

    access_cost = 0.0
    for location, identifier in location_mapping.items():
        block = location[0]
        access_cost += access_frequency.get(block, 0) * len(identifier)

    weighted_length_cost = alpha * length_cost
    weighted_similarity_cost = beta * similarity_cost
    weighted_access_cost = gamma * access_cost
    total_cost = (
        weighted_length_cost
        + weighted_similarity_cost
        + weighted_access_cost
    )

    return {
        "length_cost": length_cost,
        "similarity_cost": similarity_cost,
        "access_cost": access_cost,
        "weighted_length_cost": weighted_length_cost,
        "weighted_similarity_cost": weighted_similarity_cost,
        "weighted_access_cost": weighted_access_cost,
        "total_cost": total_cost,
    }


objective_results = calculate_objective_value(
    location_mapping,
    access_frequency,
    alpha,
    beta,
    gamma,
    epsilon,
)

weighted_component_sum = (
    objective_results["weighted_length_cost"]
    + objective_results["weighted_similarity_cost"]
    + objective_results["weighted_access_cost"]
)
assert np.isclose(weighted_component_sum, objective_results["total_cost"])

print("Objective function breakdown:")
print(
    f"  Weighted length cost (alpha times length sum): "
    f"{objective_results['weighted_length_cost']:.2f}"
)
print(
    f"  Weighted similarity cost (beta times pairwise sum): "
    f"{objective_results['weighted_similarity_cost']:.2f}"
)
print(
    f"  Weighted access cost (gamma times access sum): "
    f"{objective_results['weighted_access_cost']:.2f}"
)
print(f"  Total score: {objective_results['total_cost']:.2f}")

## Visualization of Results

The plots summarize the candidate mapping from several perspectives: identifier lengths, location counts by block and tier, and the weighted objective contributions. The component plot uses the weighted values because those are the quantities that add to the total score. The objective is a mathematical score, not a physical measurement, so its vertical axis is labeled in score units.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# The length distribution shows whether identifiers vary in size.
lengths = [len(identifier) for identifier in location_mapping.values()]
axes[0, 0].hist(
    lengths,
    bins=10,
    alpha=0.7,
    color="skyblue",
    edgecolor="black",
)
axes[0, 0].set_title("Identifier Length Distribution")
axes[0, 0].set_xlabel("Identifier length (characters)")
axes[0, 0].set_ylabel("Number of locations")
mean_length = np.mean(lengths)
axes[0, 0].axvline(
    x=mean_length,
    color="red",
    linestyle="--",
    label=f"Mean: {mean_length:.1f} characters",
)
axes[0, 0].legend()

# Block counts describe how the yard locations are distributed.
block_counts = {}
for location in location_mapping:
    block = location[0]
    block_counts[block] = block_counts.get(block, 0) + 1

axes[0, 1].bar(
    list(block_counts.keys()),
    list(block_counts.values()),
    color="lightcoral",
)
axes[0, 1].set_title("Locations per Block")
axes[0, 1].set_xlabel("Block")
axes[0, 1].set_ylabel("Number of locations")

# Tier counts show how many locations occupy each vertical level.
tier_counts = {}
for location in location_mapping:
    tier = location[3]
    tier_counts[tier] = tier_counts.get(tier, 0) + 1

axes[1, 0].bar(
    list(tier_counts.keys()),
    list(tier_counts.values()),
    color="gold",
)
axes[1, 0].set_title("Locations per Tier")
axes[1, 0].set_xlabel("Tier")
axes[1, 0].set_ylabel("Number of locations")

# Weighted components can be compared directly because they sum to the score.
components = ["Length", "Similarity", "Access"]
weighted_values = [
    objective_results["weighted_length_cost"],
    objective_results["weighted_similarity_cost"],
    objective_results["weighted_access_cost"],
]
axes[1, 1].bar(
    components,
    weighted_values,
    color=["skyblue", "lightcoral", "lightgreen"],
)
axes[1, 1].set_title("Weighted Objective Components")
axes[1, 1].set_xlabel("Objective component")
axes[1, 1].set_ylabel("Contribution to objective score")
axes[1, 1].tick_params(axis="x", rotation=30)

fig.suptitle("Summary of the Generated Yard Location Identifiers")
plt.tight_layout()
plt.show()

## Sample Identifier Analysis

Inspecting a small set of identifiers makes the field structure explicit. Each identifier can be read from left to right as block, row, position, and tier. The same formatting rule is applied to every location, while the field values distinguish the physical locations.

In [ ]:
sample_locations = [
    ("A", 1, 1, 1),
    ("A", 1, 1, 2),
    ("A", 1, 2, 1),
    ("A", 1, 2, 2),
    ("B", 1, 1, 1),
    ("B", 1, 1, 2),
    ("B", 1, 2, 1),
    ("B", 1, 2, 2),
]

print("Detailed identifier analysis:")
print("=" * 64)
print(f"{'Location':<20} {'Identifier':<14} {'Length':<8} {'Format'}")
print("=" * 64)

for location in sample_locations:
    identifier = location_mapping[location]
    print(
        f"{str(location):<20} {identifier:<14} "
        f"{len(identifier):<8} B-RR-PP-T"
    )

print("\nFormat fields:")
print("  B: Block identifier")
print("  RR: Row number, zero-padded to a minimum of two digits")
print("  PP: Position number, zero-padded to a minimum of two digits")
print("  T: Tier number")

## Sensitivity Analysis

A sensitivity analysis asks how the score changes when selected model weights change. Here, the generated mapping, access frequencies, and access weight remain fixed, while $\alpha$ and $\beta$ vary over the specified values. Each table row reevaluates the same candidate mapping with the stated weights; it does not generate a new naming convention. This distinction separates sensitivity of the score from optimization of the identifiers.

In [ ]:
alpha_values = [0.5, 1.0, 2.0]
beta_values = [5.0, 10.0, 20.0]

print("Sensitivity analysis for the fixed candidate mapping:")
print("=" * 92)
print(
    f"{'Alpha':<8} {'Beta':<8} {'Total score':<14} "
    f"{'Weighted length':<18} {'Weighted similarity':<22}"
)
print("=" * 92)

for alpha_test in alpha_values:
    for beta_test in beta_values:
        test_results = calculate_objective_value(
            location_mapping,
            access_frequency,
            alpha_test,
            beta_test,
            gamma,
            epsilon,
        )
        print(
            f"{alpha_test:<8.1f} {beta_test:<8.1f} "
            f"{test_results['total_cost']:<14.2f} "
            f"{test_results['weighted_length_cost']:<18.2f} "
            f"{test_results['weighted_similarity_cost']:<22.2f}"
        )

## Summary and Results

The notebook constructs a candidate yard-location naming convention and evaluates it against the stated constraints and objective components. The verification output reports whether identifiers are unique, whether the maximum length is within its limit, and whether the smallest pairwise Levenshtein distance meets the threshold. The objective breakdown shows how length, pairwise similarity, and block-level access frequency contribute to the candidate's score.

The sensitivity table varies the length and similarity weights while keeping the identifiers fixed. It therefore illustrates how model preferences affect the score, not how a search procedure would change the naming convention. The results should be interpreted within the model's scope: a low score is meaningful only relative to the chosen weights, distance definition, access-frequency data, and assumptions. The formulation provides a structured way to assess a naming convention, while operational deployment would also require checking local signage practices, communication needs, and the range of location numbers.